<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model03_Previous_Application_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

Mounted at /content/drive


In [3]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"

In [4]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"

In [5]:
prev = pd.read_csv(DATA_PATH + "previous_application.csv")
print("Previous application shape:", prev.shape)
display(prev.head())

Previous application shape: (1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
prev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

In [7]:
print("Data type distribution:")
display(prev.dtypes.value_counts())

Data type distribution:


,count
object,16
float64,15
int64,6


In [8]:
print("Duplicate rows:", prev.duplicated().sum())

Duplicate rows: 0


In [10]:
#Unique applicants
print("Total rows:", len(prev))
print("Unique SK_ID_CURR:", prev["SK_ID_CURR"].nunique())

Total rows: 1670214
Unique SK_ID_CURR: 338857


In [11]:
#Applications per customer
applications_per_customer = prev.groupby("SK_ID_CURR").size()
print("Applications per customer:")
display(applications_per_customer.describe())

Applications per customer:


,0
count,338857.000000
mean,4.928964
std,4.220716
min,1.000000
25%,2.000000
50%,4.000000
75%,7.000000
max,77.000000


In [12]:
print("Number of applicants with each history count:")
display(applications_per_customer.value_counts().sort_index().head(20))

Number of applicants with each history count:


,count
1,60458
2,52737
3,45966
4,38159
5,30886
6,24588
7,19216
8,15085
9,11627
10,9063


In [14]:
#Applicants with zero previous history
app_train_ids = pd.read_csv(DATA_PATH + "application_train.csv", usecols=["SK_ID_CURR"])

ids_with_prev = set(prev["SK_ID_CURR"].unique())
ids_without_prev = set(app_train_ids["SK_ID_CURR"]) - ids_with_prev

print("Applicants with NO previous_application history:", len(ids_without_prev))
print("Percentage of application_train applicants:",
      f"{len(ids_without_prev) / len(app_train_ids) * 100:.2f}%")

Applicants with NO previous_application history: 16454
Percentage of application_train applicants: 5.35%


In [15]:
missing_summary = pd.DataFrame({
    "Missing_Count": prev.isna().sum(),
    "Missing_Percentage": prev.isna().mean() * 100
})

missing_summary = missing_summary.sort_values("Missing_Percentage", ascending=False)

print("Missing-value summary:")
display(missing_summary)

Missing-value summary:


,Missing_Count,Missing_Percentage
RATE_INTEREST_PRIVILEGED,1664263,99.643698
RATE_INTEREST_PRIMARY,1664263,99.643698
AMT_DOWN_PAYMENT,895844,53.636480
RATE_DOWN_PAYMENT,895844,53.636480
NAME_TYPE_SUITE,820405,49.119754
DAYS_TERMINATION,673065,40.298129
DAYS_FIRST_DRAWING,673065,40.298129
DAYS_FIRST_DUE,673065,40.298129
DAYS_LAST_DUE_1ST_VERSION,673065,40.298129
DAYS_LAST_DUE,673065,40.298129


In [16]:
numeric_cols = prev.select_dtypes(include=np.number).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("\nNumeric columns:")
for col in numeric_cols:
    print("-", col)

Number of numeric columns: 21

Numeric columns:
- SK_ID_PREV
- SK_ID_CURR
- AMT_ANNUITY
- AMT_APPLICATION
- AMT_CREDIT
- AMT_DOWN_PAYMENT
- AMT_GOODS_PRICE
- HOUR_APPR_PROCESS_START
- NFLAG_LAST_APPL_IN_DAY
- RATE_DOWN_PAYMENT
- RATE_INTEREST_PRIMARY
- RATE_INTEREST_PRIVILEGED
- DAYS_DECISION
- SELLERPLACE_AREA
- CNT_PAYMENT
- DAYS_FIRST_DRAWING
- DAYS_FIRST_DUE
- DAYS_LAST_DUE_1ST_VERSION
- DAYS_LAST_DUE
- DAYS_TERMINATION
- NFLAG_INSURED_ON_APPROVAL


In [17]:
categorical_cols = prev.select_dtypes(include=["object"]).columns.tolist()

print("Number of categorical columns:", len(categorical_cols))
print("\nCategorical columns:")
for col in categorical_cols:
    print("-", col)

Number of categorical columns: 16

Categorical columns:
- NAME_CONTRACT_TYPE
- WEEKDAY_APPR_PROCESS_START
- FLAG_LAST_APPL_PER_CONTRACT
- NAME_CASH_LOAN_PURPOSE
- NAME_CONTRACT_STATUS
- NAME_PAYMENT_TYPE
- CODE_REJECT_REASON
- NAME_TYPE_SUITE
- NAME_CLIENT_TYPE
- NAME_GOODS_CATEGORY
- NAME_PORTFOLIO
- NAME_PRODUCT_TYPE
- CHANNEL_TYPE
- NAME_SELLER_INDUSTRY
- NAME_YIELD_GROUP
- PRODUCT_COMBINATION


In [18]:
print("Numeric descriptive statistics:")
display(prev[numeric_cols].describe().T)

Numeric descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,1670214.0,1.923089e+06,532597.958696,1.000001e+06,1.461857e+06,1.923110e+06,2.384280e+06,2845382.000
SK_ID_CURR,1670214.0,2.783572e+05,102814.823849,1.000010e+05,1.893290e+05,2.787145e+05,3.675140e+05,456255.000
AMT_ANNUITY,1297979.0,1.595512e+04,14782.137335,0.000000e+00,6.321780e+03,1.125000e+04,2.065842e+04,418058.145
AMT_APPLICATION,1670214.0,1.752339e+05,292779.762387,0.000000e+00,1.872000e+04,7.104600e+04,1.803600e+05,6905160.000
AMT_CREDIT,1670213.0,1.961140e+05,318574.616546,0.000000e+00,2.416050e+04,8.054100e+04,2.164185e+05,6905160.000
AMT_DOWN_PAYMENT,774370.0,6.697402e+03,20921.495410,-9.000000e-01,0.000000e+00,1.638000e+03,7.740000e+03,3060045.000
AMT_GOODS_PRICE,1284699.0,2.278473e+05,315396.557937,0.000000e+00,5.084100e+04,1.123200e+05,2.340000e+05,6905160.000
HOUR_APPR_PROCESS_START,1670214.0,1.248418e+01,3.334028,0.000000e+00,1.000000e+01,1.200000e+01,1.500000e+01,23.000
NFLAG_LAST_APPL_IN_DAY,1670214.0,9.964675e-01,0.059330,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000
RATE_DOWN_PAYMENT,774370.0,7.963682e-02,0.107823,-1.497876e-05,0.000000e+00,5.160508e-02,1.089091e-01,1.000


In [19]:
print("Categorical value distributions:")
for col in categorical_cols:
    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)
    display(prev[col].value_counts(dropna=False).head(20))

Categorical value distributions:

NAME_CONTRACT_TYPE


,count
NAME_CONTRACT_TYPE,
Cash loans,747553
Consumer loans,729151
Revolving loans,193164
XNA,346



WEEKDAY_APPR_PROCESS_START


,count
WEEKDAY_APPR_PROCESS_START,
TUESDAY,255118
WEDNESDAY,255010
MONDAY,253557
FRIDAY,252048
THURSDAY,249099
SATURDAY,240631
SUNDAY,164751



FLAG_LAST_APPL_PER_CONTRACT


...

,count
NAME_SELLER_INDUSTRY,
XNA,855720
Consumer electronics,398265
Connectivity,276029
Furniture,57849
Construction,29781
Clothing,23949
Industry,19194
Auto technology,4990
Jewelry,2709



NAME_YIELD_GROUP


,count
NAME_YIELD_GROUP,
XNA,517215
middle,385532
high,353331
low_normal,322095
low_action,92041



PRODUCT_COMBINATION


,count
PRODUCT_COMBINATION,
Cash,285990
POS household with interest,263622
POS mobile with interest,220670
Cash X-Sell: middle,143883
Cash X-Sell: low,130248
Card Street,112582
POS industry with interest,98833
POS household without interest,82908
Card X-Sell,80582


In [20]:
financial_cols = [c for c in [
    "AMT_CREDIT", "AMT_ANNUITY", "AMT_APPLICATION",
    "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT"
] if c in prev.columns]

print("Financial columns found:", financial_cols)

if financial_cols:
    display(prev[financial_cols].describe().T)

Financial columns found: ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'RATE_DOWN_PAYMENT']


,count,mean,std,min,25%,50%,75%,max
AMT_CREDIT,1670213.0,196114.021218,318574.616546,0.000000,24160.50,80541.000000,216418.500000,6905160.000
AMT_ANNUITY,1297979.0,15955.120659,14782.137335,0.000000,6321.78,11250.000000,20658.420000,418058.145
AMT_APPLICATION,1670214.0,175233.860360,292779.762387,0.000000,18720.00,71046.000000,180360.000000,6905160.000
AMT_DOWN_PAYMENT,774370.0,6697.402139,20921.495410,-0.900000,0.00,1638.000000,7740.000000,3060045.000
AMT_GOODS_PRICE,1284699.0,227847.279283,315396.557937,0.000000,50841.00,112320.000000,234000.000000,6905160.000
RATE_DOWN_PAYMENT,774370.0,0.079637,0.107823,-0.000015,0.00,0.051605,0.108909,1.000


In [21]:
status_cols = [c for c in [
    "NAME_CONTRACT_STATUS", "NAME_CONTRACT_TYPE", "NAME_CLIENT_TYPE",
    "NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE"
] if c in prev.columns]

print("Important categorical behavior columns:", status_cols)

for col in status_cols:
    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)
    display(prev[col].value_counts(dropna=False).head(20))

Important categorical behavior columns: ['NAME_CONTRACT_STATUS', 'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE']

NAME_CONTRACT_STATUS


,count
NAME_CONTRACT_STATUS,
Approved,1036781
Canceled,316319
Refused,290678
Unused offer,26436



NAME_CONTRACT_TYPE


,count
NAME_CONTRACT_TYPE,
Cash loans,747553
Consumer loans,729151
Revolving loans,193164
XNA,346



NAME_CLIENT_TYPE


...

,count
NAME_PORTFOLIO,
POS,691011
Cash,461563
XNA,372230
Cards,144985
Cars,425



NAME_PRODUCT_TYPE


,count
NAME_PRODUCT_TYPE,
XNA,1063666
x-sell,456287
walk-in,150261



CHANNEL_TYPE


,count
CHANNEL_TYPE,
Credit and cash offices,719968
Country-wide,494690
Stone,212083
Regional / Local,108528
Contact center,71297
AP+ (Cash loan),57046
Channel of corporate sales,6150
Car dealer,452


In [22]:
if "DAYS_DECISION" in prev.columns:
    print("DAYS_DECISION statistics:")
    display(prev["DAYS_DECISION"].describe())

DAYS_DECISION statistics:


,DAYS_DECISION
count,1.670214e+06
mean,-8.806797e+02
std,7.790997e+02
min,-2.922000e+03
25%,-1.300000e+03
50%,-5.810000e+02
75%,-2.800000e+02
max,-1.000000e+00


In [23]:
if "CNT_PAYMENT" in prev.columns:
    print("CNT_PAYMENT statistics:")
    display(prev["CNT_PAYMENT"].describe())

CNT_PAYMENT statistics:


,CNT_PAYMENT
count,1.297984e+06
mean,1.605408e+01
std,1.456729e+01
min,0.000000e+00
25%,6.000000e+00
50%,1.200000e+01
75%,2.400000e+01
max,8.400000e+01


In [24]:
if "AMT_APPLICATION" in prev.columns and "AMT_CREDIT" in prev.columns:
    print("Application amount vs credit amount:")
    display(prev[["AMT_APPLICATION", "AMT_CREDIT"]].describe())

Application amount vs credit amount:


,AMT_APPLICATION,AMT_CREDIT
count,1.670214e+06,1.670213e+06
mean,1.752339e+05,1.961140e+05
std,2.927798e+05,3.185746e+05
min,0.000000e+00,0.000000e+00
25%,1.872000e+04,2.416050e+04
50%,7.104600e+04,8.054100e+04
75%,1.803600e+05,2.164185e+05
max,6.905160e+06,6.905160e+06


In [25]:
if "NAME_CONTRACT_STATUS" in prev.columns:
    status_counts_per_customer = prev.groupby("SK_ID_CURR")["NAME_CONTRACT_STATUS"].nunique()
    print("Distribution of number of different contract statuses per applicant:")
    display(status_counts_per_customer.value_counts().sort_index())

Distribution of number of different contract statuses per applicant:


,count
NAME_CONTRACT_STATUS,
1,145544
2,119997
3,68054
4,5262


In [26]:
example_customer = applications_per_customer.idxmax()
print("Example applicant with the most previous applications:", example_customer)
display(prev[prev["SK_ID_CURR"] == example_customer])

Example applicant with the most previous applications: 187868


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
37305,1703693,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,WEDNESDAY,2,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
56117,1677569,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,SATURDAY,9,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
67781,1651393,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,SATURDAY,7,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
92787,1173037,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,THURSDAY,9,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
170959,2636986,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,WEDNESDAY,8,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552993,2455456,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,MONDAY,6,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
1575005,1753307,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,TUESDAY,4,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
1581066,1404221,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,FRIDAY,3,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
1636755,2515824,187868,Cash loans,NaN,0.0,0.0,NaN,NaN,WEDNESDAY,2,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
